## Azure AI Foundry SDK Hands-on
This notebook demonstrates how to use the Azure AI Foundry SDK for AI applications, providing enhanced capabilities over traditional Azure OpenAI integration.

## Overview
This hands-on session helps you understand Azure AI Foundry SDK integration and text analysis capabilities.

The Azure AI Foundry SDK provides a unified interface for working with Azure AI services, offering improved connection management, monitoring, and collaboration features compared to direct OpenAI API usage.

For more information about Azure AI Foundry, refer to the [official documentation](https://learn.microsoft.com/azure/ai-studio/)

#### Prerequisite
Please complete the **00_Setup.ipynb** notebook before running this notebook.

## Table of Contents

[Overview](#overview)  
[Getting started with Azure AI Foundry SDK](#getting-started-with-azure-ai-foundry-sdk)  
[Build your first prompt with Foundry](#build-your-first-prompt-with-foundry)  

[Use Cases](#use-cases)  
[1. Summarize Text](#summarize-text)  
[2. Classify Text](#classify-text)  
[3. Generate New Product Names](#generate-new-product-names)  
[4. Embeddings with Foundry](#embeddings-with-foundry)  
[5. Working with Multiple Models](#working-with-multiple-models)  

[References](#references)

### Build your first prompt with Azure AI Foundry
This exercise provides a basic introduction for submitting prompts using the Azure AI Foundry SDK.

**Steps you will complete**:

1. Install Azure AI Foundry SDK and dependencies
2. Load standard helper libraries and establish Foundry connection
3. Connect to your deployed models through Foundry
4. Create a simple prompt for the model
5. Submit your request using the Foundry SDK
6. Monitor and track your API usage

#### Azure Authentication - see setup notebook for explanation. You may need to rerun this if the credetials expire

In [6]:
# Azure Authentication using Helper Module
import os
from dotenv import load_dotenv
from azure_auth_helper import authenticate_azure

# Load environment variables
load_dotenv("./.env")

# Get tenant ID from environment variables
TENANT_ID = os.getenv('AZURE_TENANT_ID')
if not TENANT_ID:
    raise ValueError("AZURE_TENANT_ID not found in .env file. Please add it to your .env file.")

print(f"🏢 Using tenant ID: {TENANT_ID}")

# Authenticate with Azure using browser authentication (interactive)
# Opens browser window for Azure login with specific tenant
credential = authenticate_azure(
    auth_method='browser', 
    tenant_id=TENANT_ID
)

# Test that the credential actually works
print(f"🔍 Testing credential type: {type(credential).__name__}")
try:
    # Try to get a token to validate the credential
    token = credential.get_token("https://management.azure.com/.default")
    print("✅ Credential test successful!")
    print(f"Token expires: {token.expires_on}")
except Exception as e:
    print(f"❌ Credential test failed: {e}")
    print(f"   Error type: {type(e).__name__}")
    raise

print("🎉 Ready to use Azure AI Foundry!")

🏢 Using tenant ID: 7ec824c6-48d9-4e32-b7a1-9a3df8bdcf66


DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: Authentication failed: AADSTS700016: Application with identifier 'ffbf62a3-8d06-4708-97e6-4d5a00cb1cc3' was not found in the directory 'Contoso'. This can happen if the application has not been installed by the administrator of the tenant or consented to by any user in the tenant. You may have sent your authentication request to the wrong tenant. Trace ID: 570349ed-e6d3-4f88-ac14-b63bcac20500 Correlation ID: 266ad276-beb8-4750-a8a6-a2f61ced31e3 Timestamp: 2025-08-27 08:04:46Z
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/python/identity/defaultazurecredential/troubleshoot.


ℹ️ No existing credentials found, will authenticate...
🌐 Starting interactive browser authentication...
🏢 Authenticating with tenant: 7ec824c6-48d9-4e32-b7a1-9a3df8bdcf66
✅ Authentication successful!
🔍 Testing credential type: InteractiveBrowserCredential
✅ Credential test successful!
Token expires: 1756285694
🎉 Ready to use Azure AI Foundry!
✅ Authentication successful!
🔍 Testing credential type: InteractiveBrowserCredential
✅ Credential test successful!
Token expires: 1756285694
🎉 Ready to use Azure AI Foundry!


### 2. Import helper libraries and establish Foundry connection

In [13]:
import os
import numpy as np
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from datetime import datetime

# Load environment variables
load_dotenv("./.env")

project = AIProjectClient(
    endpoint=os.getenv("FOUNDRY_API_ENDPOINT"),
    credential=credential,
)

print("✅ Successfully connected to Azure AI Foundry")
print(f"📊 Project endpoint: {os.getenv('FOUNDRY_API_ENDPOINT')}")
print(f"🏢 Resource Group: {os.getenv('AZURE_RESOURCE_GROUP')}")
print(f"🔑 Using browser credential for project authentication")

✅ Successfully connected to Azure AI Foundry
📊 Project endpoint: https://hannahhowell-3776-resource.services.ai.azure.com/api/projects/hannahhowell-3776
🏢 Resource Group: rg-hannahhowell-3776
🔑 Using browser credential for project authentication


### 3. Get deployed model

In [14]:
# Get model deployment names from environment
gpt4o_model = os.getenv('GPT4O_DEPLOYMENT_NAME', 'gpt-4o')

## 4. Prompt Design  

"The magic of large language models is that by being trained to minimize this prediction error over vast quantities of text, the models end up learning concepts useful for these predictions. For example, they learn concepts like"(1):

* how to spell
* how grammar works
* how to paraphrase
* how to answer questions
* how to hold a conversation
* how to write in many languages
* how to code
* etc.

#### How to control a large language model  
"Of all the inputs to a large language model, by far the most influential is the text prompt"

Large language models can be prompted to produce output in a few ways:

- Instruction: Tell the model what you want
- Completion: Induce the model to complete the beginning of what you want
- Demonstration: Show the model what you want, with either:
  - A few examples in the prompt
  - Many hundreds or thousands of examples in a fine-tuning training dataset

#### System vs User Prompts

**System Prompt**: Sets the overall behavior, role, and context for the AI assistant. This message:
- Defines the AI's persona and capabilities
- Establishes rules and guidelines for responses
- Sets the tone and style of interaction
- Remains consistent throughout the conversation
- Example: "You are a helpful assistant that explains complex topics in simple terms."

**User Prompt**: Contains the specific question or task from the user. This message:
- Provides the actual request or query
- Can change with each interaction
- Contains the specific information to process
- Example: "Explain how machine learning works."

**Best Practice**: Define your system prompt clearly to establish consistent behavior, then use user prompts for specific requests.

#### There are three basic guidelines to creating prompts:

**Show and tell**. Make it clear what you want either through instructions, examples, or a combination of the two. If you want the model to rank a list of items in alphabetical order or to classify a paragraph by sentiment, show it that's what you want.

**Provide quality data**. If you're trying to build a classifier or get the model to follow a pattern, make sure that there are enough examples. Be sure to proofread your examples — the model is usually smart enough to see through basic spelling mistakes and give you a response, but it also might assume this is intentional and it can affect the response.

**Check your settings.** The temperature and top_p settings control how deterministic the model is in generating a response. If you're asking it for a response where there's only one right answer, then you'd want to set these lower. If you're looking for more diverse responses, then you might want to set them higher. The number one mistake people make with these settings is assuming that they're "cleverness" or "creativity" controls.

Source: https://github.com/Azure/OpenAI/blob/main/How%20to/Completions.md

In [15]:
# Define system prompt separately for clarity
system_prompt = "You are a helpful assistant that provides clear, concise answers to questions."

# Create your first prompt
text_prompt = "Should oxford commas always be used?"

# Now we can use the Foundry project client to get the OpenAI client
# This gives us all the Foundry benefits like monitoring and tracking
models = project.get_openai_client(api_version="2024-10-21")

response = models.chat.completions.create(
    model=gpt4o_model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text_prompt}
    ]
)

response.choices[0].message.content

'The use of the Oxford comma (the comma before "and" or "or" in a list) is a style choice, but it is generally recommended in situations where it prevents ambiguity or enhances clarity.\n\n### Pros of Using the Oxford Comma:\n- **Avoids confusion**: It clarifies meaning in sentences where items in a list could be misinterpreted.\n   - Example without Oxford comma: "I love my parents, Taylor Swift and Harry Styles." (This could imply your parents are Taylor Swift and Harry Styles.)\n   - Example with Oxford comma: "I love my parents, Taylor Swift, and Harry Styles." (Clearly separates the items.)\n- **Consistent style**: Using the comma consistently eliminates guesswork.\n\n### Exceptions:\n- Some style guides, like Associated Press (AP), generally omit it unless it is needed for clarity. Others, like the Chicago Manual of Style, recommend its consistent use.\n\nIn summary, the Oxford comma should be used when it improves clarity, and many prefer to use it consistently to avoid confusio

### Repeat the same call, how do the results compare?
GenerativeAI is probabilistic so the same inputs often give different but similar outputs.

In [16]:
# Repeat the same call to see variation in responses
response = models.chat.completions.create(
    model=gpt4o_model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text_prompt}
    ]
)

response.choices[0].message.content

'Not always, but the Oxford comma—used before the final item in a list (e.g., "apples, oranges, and bananas")—is recommended in many cases for clarity. Its use can prevent ambiguity or misinterpretation, especially in complex or legal writing. However, some style guides, like the Associated Press (AP), omit it unless needed for clarity. Whether to use it often depends on the context and the style guide you\'re following.'

## Use Cases

### 1. Summarize Text
Let's use Azure AI Foundry to summarize a piece of text with enhanced monitoring.

In [ ]:
# Define system prompt for summarization task
summarization_system_prompt = "You are an expert at creating concise, accurate summaries. Provide clear 2-3 sentence summaries that capture the key points."

# Text to summarize
text_to_summarize = """
Azure AI Foundry is Microsoft's comprehensive platform for building, deploying, and managing AI applications. 
It provides a unified experience for data scientists, developers, and business users to collaborate on AI projects. 
The platform includes capabilities for model development, deployment, monitoring, and governance. 
With Azure AI Foundry, organizations can accelerate their AI journey while maintaining security, compliance, and ethical AI practices. 
The platform supports various AI workloads including natural language processing, computer vision, and machine learning.
"""

summarize_prompt = f"Please summarize the following text:\n\n{text_to_summarize}"

# Get summary using Foundry client
response = models.chat.completions.create(
    model=gpt4o_model,
    messages=[
        {"role": "system", "content": summarization_system_prompt},
        {"role": "user", "content": summarize_prompt}
    ],
    temperature=0.3
)

summary = response.choices[0].message.content
print("📝 **Summary:**")
print(summary)

📝 **Summary:**
Azure AI Foundry is Microsoft's all-in-one platform for creating, deploying, and managing AI applications, offering tools for model development, monitoring, and governance. It enables collaboration among data scientists, developers, and business users while supporting diverse AI workloads like natural language processing, computer vision, and machine learning. The platform helps organizations advance their AI initiatives securely and ethically.


### 2. Classify Text
Demonstrate text classification using Azure AI Foundry with enhanced error handling.

In [17]:
# Define system prompt for classification task
classification_system_prompt = "You are a sentiment analysis expert. Classify text into exactly one category: POSITIVE, NEGATIVE, or NEUTRAL. Respond with only the category name."

# Text classification example
texts_to_classify = [
    "I love this new smartphone! The camera quality is amazing.",
    "The delivery was delayed and the package was damaged.",
    "The weather forecast shows rain for the next three days."
]

classify_prompt_template = """
Classify the following text:

Text: {text}
Category:
"""

print("🏷️ **Text Classification Results:**")
print()

for i, text in enumerate(texts_to_classify, 1):
    user_prompt = classify_prompt_template.format(text=text)
    
    response = models.chat.completions.create(
        model=gpt4o_model,
        messages=[
            {"role": "system", "content": classification_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=10
    )
    
    classification = response.choices[0].message.content.strip()
    print(f"**Text {i}:** {text}")
    print(f"**Classification:** {classification}")
    print()

🏷️ **Text Classification Results:**

**Text 1:** I love this new smartphone! The camera quality is amazing.
**Classification:** POSITIVE

**Text 1:** I love this new smartphone! The camera quality is amazing.
**Classification:** POSITIVE

**Text 2:** The delivery was delayed and the package was damaged.
**Classification:** NEGATIVE

**Text 2:** The delivery was delayed and the package was damaged.
**Classification:** NEGATIVE

**Text 3:** The weather forecast shows rain for the next three days.
**Classification:** NEUTRAL

**Text 3:** The weather forecast shows rain for the next three days.
**Classification:** NEUTRAL



### 3. Generate New Product Names
Use Azure AI Foundry for creative content generation with multiple variations.

In [18]:
# Define system prompt for creative generation
creative_system_prompt = "You are a creative marketing expert who specializes in generating memorable, catchy product names. Focus on names that are marketable and reflect the product's unique features."

# Product name generation
product_description = "An eco-friendly water bottle made from recycled materials with temperature control features"

generation_prompt = f"""
Generate 5 creative and catchy product names for the following product:

Product: {product_description}

Requirements:
- Names should be memorable and marketable
- Reflect the eco-friendly and innovative nature
- Keep names under 20 characters

Product Names:
"""

response = models.chat.completions.create(
    model=gpt4o_model,
    messages=[
        {"role": "system", "content": creative_system_prompt},
        {"role": "user", "content": generation_prompt}
    ],
    temperature=0.8,
    max_tokens=200
)

product_names = response.choices[0].message.content
print("🚀 **Generated Product Names:**")
print(product_names)

🚀 **Generated Product Names:**
1. **EcoTemp Bottle**  
2. **RecyCool**  
3. **GreenChill**  
4. **SustainSip**  
5. **EarthThermo**  


## References

- [Azure AI Foundry Documentation](https://learn.microsoft.com/azure/ai-studio/)
- [Azure AI Foundry SDK Reference](https://learn.microsoft.com/python/api/overview/azure/ai-ml-readme)
- [Azure OpenAI Service](https://learn.microsoft.com/azure/ai-services/openai/)
- [Prompt Engineering Guide](https://learn.microsoft.com/azure/ai-services/openai/concepts/prompt-engineering)
- [Azure AI Safety and Responsible AI](https://learn.microsoft.com/azure/ai-services/responsible-use-of-ai-overview)